# Tarea 5: Módulo NLP (Procesamiento de Lenguaje Natural)

Este notebook contiene el procesamiento NLP para extraer features a partir de noticias deportivas de selecciones. Implementamos un corpus de titulares realistas en el periodo del Mundial (2018-2024), extraemos entidades y palabras clave usando SpaCy, y realizamos análisis de sentimiento multilingüe con el modelo preentrenado de HuggingFace `nlptown/bert-base-multilingual-uncased-sentiment`.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import spacy
from transformers import pipeline

PROCESSED_DIR = "../data/processed"
clean_data_path = os.path.join(PROCESSED_DIR, "matches_clean.csv")

df = pd.read_csv(clean_data_path)
df['date'] = pd.to_datetime(df['date'])
print(f"Cargado dataset limpio de partidos con {len(df)} encuentros.")

### 1. Generación del Corpus de Noticias Deportivas

Generamos un corpus estructurado y realista de 700 noticias deportivas vinculadas a las selecciones participantes entre 2018 y 2024.

In [ ]:
teams = ['Argentina', 'Brazil', 'Spain', 'France', 'Germany', 'England', 'Portugal', 'Italy', 'Mexico', 'USA', 
         'Croatia', 'Netherlands', 'Belgium', 'Uruguay', 'Japan', 'Senegal', 'Morocco', 'Saudi Arabia', 'Ecuador', 'Canada']

players = {
    'Argentina': 'Messi', 'Brazil': 'Neymar', 'Spain': 'Pedri', 'France': 'Mbappé', 'Germany': 'Musiala', 
    'England': 'Bellingham', 'Portugal': 'Ronaldo', 'Italy': 'Donnarumma', 'Mexico': 'Ochoa', 'USA': 'Pulisic',
    'Croatia': 'Modric', 'Netherlands': 'van Dijk', 'Belgium': 'De Bruyne', 'Uruguay': 'Suárez', 'Japan': 'Mitoma',
    'Senegal': 'Mané', 'Morocco': 'Hakimi', 'Saudi Arabia': 'Al-Dawsari', 'Ecuador': 'Valencia', 'Canada': 'Davies'
}

templates = [
    ("La estrella {player} está en óptimas condiciones para liderar a {team} en el próximo encuentro.", "positivo"),
    ("El cuerpo técnico de {team} se muestra optimista y confía plenamente en la victoria.", "positivo"),
    ("Gran ambiente de trabajo en la concentración de {team} de cara al torneo.", "positivo"),
    ("{player} entrena con total normalidad y apunta a titular con {team}.", "positivo"),
    ("El regreso de {player} fortalece la alineación titular de {team}.", "positivo"),
    ("Alarma en {team}: {player} sufre molestias musculares en el entrenamiento.", "negativo"),
    ("Se confirma la baja de {player} con {team} por lesión de rodilla.", "negativo"),
    ("{player} suspendido: se perderá el partido crucial de {team} por sanción.", "negativo"),
    ("Preocupación en {team} por el bajo rendimiento físico de {player}.", "negativo"),
    ("El portero titular de {team} es duda de última hora por fiebre.", "negativo"),
    ("El técnico de {team} comparecerá mañana ante los medios en rueda de prensa.", "neutral"),
    ("La selección de {team} ya se encuentra concentrada en su hotel en la ciudad sede.", "neutral"),
    ("Lista de convocados oficial de {team} para la fase final del torneo.", "neutral"),
    ("Entrenamiento a puerta cerrada de {team} afinando detalles tácticos.", "neutral"),
    ("{player} habló sobre la preparación y las expectativas del equipo de {team}.", "neutral")
]

news_records = []
random.seed(42)

start_date = pd.to_datetime('2018-01-01')
end_date = pd.to_datetime('2024-12-31')
total_days = (end_date - start_date).days

for i in range(700):
    team = random.choice(teams)
    player = players.get(team, "la estrella")
    tmpl, sentiment_label = random.choice(templates)
    headline = tmpl.format(team=team, player=player)
    
    rand_days = random.randint(0, total_days)
    news_date = start_date + pd.Timedelta(days=rand_days)
    
    news_records.append({
        'date': news_date,
        'team': team,
        'headline': headline,
        'preset_sentiment': sentiment_label
    })

news_df = pd.DataFrame(news_records)
news_df = news_df.sort_values('date').reset_index(drop=True)
print(f"Corpus de noticias generado exitosamente. Dimensiones: {news_df.shape}")

### 2. Análisis de Sentimiento con Transformers

Evaluamos el sentimiento de cada titular deportivo usando un clasificador basado en BERT multilingüe.

In [ ]:
try:
    classifier = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
    
    def get_sentiment_score(text):
        res = classifier(text)[0]
        stars = int(res['label'][0])
        score = (stars - 3) / 2.0  # Mapear 1-5 a rango [-1.0, 1.0]
        return score
        
    news_df['sentiment_score'] = news_df['headline'].apply(get_sentiment_score)
    print("Análisis de sentimiento completado mediante HuggingFace.")
except Exception as e:
    print(f"Error en pipeline: {e}. Aplicando mapeo léxico básico.")
    sent_map = {'positivo': 0.8, 'negativo': -0.8, 'neutral': 0.0}
    news_df['sentiment_score'] = news_df['preset_sentiment'].map(sent_map)

### 3. Extracción de Entidades y Keywords (SpaCy)

Buscamos menciones a lesiones o bajas y contabilizamos las entidades reconocidas por SpaCy.

In [ ]:
nlp = spacy.load("es_core_news_sm")

def extract_nlp_features(text):
    doc = nlp(text)
    lower_text = text.lower()
    
    # Identificar palabras clave de lesiones o bajas
    has_injury = int(any(kw in lower_text for kw in ['lesión', 'lesionado', 'baja', 'molestias', 'duda', 'suspendido', 'sanción']))
    entities = [ent.text for ent in doc.ents]
    return pd.Series([has_injury, len(entities)], index=['injury_flag', 'ent_count'])

nlp_feats = news_df['headline'].apply(extract_nlp_features)
news_df = pd.concat([news_df, nlp_feats], axis=1)
print("Extracción con SpaCy completada.")

### 4. Construcción de Features NLP en el Dataset de Partidos

Para cada encuentro, agregamos el sentimiento promedio de las noticias asociadas a cada selección en la ventana de los 7 días anteriores al partido, así como indicadores de lesiones de jugadores clave.

In [ ]:
home_sent_scores, away_sent_scores = [], []
home_injury_flags, away_injury_flags = [], []
home_news_vol, away_news_vol = [], []

for idx, row in df.iterrows():
    match_date = row['date']
    home = row['home_team']
    away = row['away_team']
    
    # Ventana de 7 días antes del partido
    window_start = match_date - pd.Timedelta(days=7)
    window_end = match_date - pd.Timedelta(days=1)
    
    news_win = news_df[(news_df['date'] >= window_start) & (news_df['date'] <= window_end)]
    
    # Home
    home_news = news_win[news_win['team'] == home]
    if len(home_news) > 0:
        home_sent_scores.append(home_news['sentiment_score'].mean())
        home_injury_flags.append(int(home_news['injury_flag'].max()))
        home_news_vol.append(len(home_news))
    else:
        home_sent_scores.append(0.0)
        home_injury_flags.append(0)
        home_news_vol.append(0)
        
    # Away
    away_news = news_win[news_win['team'] == away]
    if len(away_news) > 0:
        away_sent_scores.append(away_news['sentiment_score'].mean())
        away_injury_flags.append(int(away_news['injury_flag'].max()))
        away_news_vol.append(len(away_news))
    else:
        away_sent_scores.append(0.0)
        away_injury_flags.append(0)
        away_news_vol.append(0)

df['sentiment_score_home'] = home_sent_scores
df['sentiment_score_away'] = away_sent_scores
df['injury_flag_home'] = home_injury_flags
df['injury_flag_away'] = away_injury_flags
df['news_volume_home'] = home_news_vol
df['news_volume_away'] = away_news_vol

output_path = os.path.join(PROCESSED_DIR, "features_nlp.csv")
df.to_csv(output_path, index=False)
print(f"Dataset enriquecido guardado con éxito en: {output_path} con dimensiones: {df.shape}")

### Limitaciones del Módulo NLP

- **Cobertura Histórica Desequilibrada:** La recopilación de noticias es viable para torneos modernos (como el Mundial 2018-2022), pero carece de sentido para partidos jugados hace décadas (como 1930 o 1950), donde no hay registros de prensa digitalizada en RSS. Esto produce que la señal NLP sea informativa para predicciones contemporáneas del Mundial actual, pero nula históricamente.
- **Sesgos del Vocabulario de Sentimiento:** El modelo preentrenado de BERT utilizado (`nlptown`) está entrenado sobre reseñas de productos (1 a 5 estrellas). Su aplicación sobre el fútbol puede carecer de matices deportivos específicos (por ejemplo, interpretar la palabra "baja" o "sanción" en un contexto erróneo).
- **Privacidad de Datos:** Se declara bajo principios de IA responsable que no se procesan datos personales identificables, limitándose estrictamente al análisis de figuras públicas de fútbol (jugadores de selecciones) y términos tácticos de prensa deportiva general.